# 04. Ein Sprachmodell bewerten

**Leitfrage:** Wie viel Wahrscheinlichkeit gibt ein Sprachmodell den Wörtern, die in neuen Reviews tatsächlich stehen?

**Arbeitsauftrag:** [Eine geglättete Wahrscheinlichkeit berechnen](#add-one) und [den Modellvergleich interpretieren](#compare). Du brauchst bedingte Wahrscheinlichkeiten und Python-Wörterbücher. Dieses Notebook lädt seine Daten selbst.

## 1. Einen Satz Wort für Wort bewerten

Wir prüfen den Beispielsatz **The special effects work.** Die Modelle lernen aus 500 Trainingsreviews. Für den späteren Vergleich reservieren wir 100 weitere Reviews. Alle Modelle verwenden dieselben Satzgrenzen und dasselbe Zielvokabular.

In [ ]:
from math import exp, log, prod
from pathlib import Path
from ngram_data import count_followers, load_sentences, word_events

data_folder = Path('data/imdb')
training_sentences = load_sentences(data_folder / 'train')
validation_sentences = load_sentences(data_folder / 'validation')

Ein Wort bekommt einen eigenen Eintrag, wenn wir es im Training mindestens zweimal zählen. Einmalige und später unbekannte Wörter stehen gemeinsam für `<unk>`. Auch `<end>` gehört zum Zielvokabular. `<start>` erscheint ausschließlich im Kontext. So kann jedes Modell dieselben Ziel-Tokens bewerten.

In [ ]:
raw_counts = count_followers(training_sentences, n=1)[()]
vocabulary = {word for word, count in raw_counts.items() if count >= 2}
vocabulary.add('<unk>')

def map_words(sentence: list[str]) -> list[str]:
    return [word if word in vocabulary else '<unk>' for word in sentence]

training = [map_words(sentence) for sentence in training_sentences]
validation = [map_words(sentence) for sentence in validation_sentences]
print('Possible next tokens:', len(vocabulary))

### 1.1 Die Faktoren der Kettenregel

Bei einem Bigramm zählt jeweils das letzte Wort. `word_events` liefert für jeden Schritt ein Paar aus Kontext und Ziel. Das Verhältnis aus passenden und allen Fortsetzungen ergibt die geschätzte Wahrscheinlichkeit. Das Produkt der fünf Faktoren schließt das Satzende ein.

In [ ]:
models = {n: count_followers(training, n) for n in (1, 2, 3)}
example = map_words('the special effects work'.split())
factors = []
print('  Before  Target      Matched / Total    P')
for context, word in word_events(example, n=2):
    counts = models[2][context]
    observed = counts.get(word, 0)
    total = sum(counts.values())
    factors.append(observed / total)
    print(f'{context[0]:>8}  {word:<8}  {observed:>3} / {total:<4}  {observed / total:.4f}')
print('Sentence probability:', prod(factors))

Jedes der vier Wörter ist bekannt. Nach `effects` hat das Modell `work` noch nie gesehen. An dieser Stelle wird ein Faktor null und damit auch das ganze Produkt.

<a id="add-one"></a>
## 2. Add-one-Glättung

**Dein Auftrag:** Ergänze die Rückgabe von `add_one_probability`. Nach einem Kontext $h$ bekommt jedes mögliche Folgewort einen zusätzlichen Zählwert. Für ein Wort $w$ ergibt sich

$$P(w\mid h)=\frac{C(h,w)+1}{C(h)+|V|}.$$

`observed` zählt das gewünschte Wort, `total` alle beobachteten Fortsetzungen und `vocabulary_size` alle möglichen Ziel-Tokens. Wie groß wird die Wahrscheinlichkeit von `work` nach `effects`?

In [ ]:
def add_one_probability(counts: dict[str, int], word: str, vocabulary_size: int) -> float:
    observed = counts.get(word, 0)
    total = sum(counts.values())
    # TODO: Add one count per possible target.
    raise NotImplementedError

In [ ]:
counts = models[2][('effects',)]
for word in ['are', 'work']:
    before = counts.get(word, 0) / sum(counts.values())
    after = add_one_probability(counts, word, len(vocabulary))
    print(f'{word:>4}: {before:.4%} before, {after:.4%} after smoothing')
assert add_one_probability(counts, 'work', len(vocabulary)) == 1 / 4975

Die Glättung verteilt viel Masse auf seltene und unbeobachtete Fortsetzungen. Nach `effects` stehen 45 echten Beobachtungen 4.930 zusätzliche Zählwerte gegenüber. Für ein ganz unbekanntes Wortpaar ist das Wörterbuch leer. Dann erhält jedes Ziel die Wahrscheinlichkeit $1/|V|$.

<a id="compare"></a>
## 3. Wie überraschend sind die Wörter einer neuen Review?

**Perplexität, die:** Maß dafür, wie überraschend die tatsächlich gelesenen Wörter eines Textes für ein Sprachmodell im Durchschnitt sind. Je mehr Wahrscheinlichkeit das Modell diesen Wörtern zuweist, desto kleiner ist die Perplexität. Ein Wert von 4 entspricht derselben mittleren Überraschung wie eine Wahl unter vier gleich wahrscheinlichen Möglichkeiten pro Schritt.

Für jedes der $n$ Ziel-Tokens lesen wir seine Wahrscheinlichkeit $p_i$ im jeweiligen Kontext ab. Die Perplexität ist der Kehrwert des geometrischen Mittels dieser Wahrscheinlichkeiten:

$$\mathrm{PP}=\frac{1}{\sqrt[n]{p_1\cdot p_2\cdots p_n}}.$$

**Beispiel:** Zwei Modelle lesen *The special effects work.* Wir geben *the special* vor und bewerten nur die folgenden beiden Wörter. Die Wahrscheinlichkeiten sind für das Rechenbeispiel gewählt.

| Modell | $P(\mathrm{effects}\mid\mathrm{the\ special})$ | $P(\mathrm{work}\mid\mathrm{the\ special\ effects})$ | Perplexität |
| --- | ---: | ---: | ---: |
| A | ½ | ½ | $1/\sqrt{1/2\cdot1/2}=2$ |
| B | ½ | ⅛ | $1/\sqrt{1/2\cdot1/8}=4$ |

Bei *effects* sind beide Modelle gleich überrascht. Für *work* gibt Modell B nur ein Viertel der Wahrscheinlichkeit von Modell A an. Dadurch verdoppelt sich über die beiden Wörter die Perplexität. Rechne die Werte mit der vorbereiteten Funktion nach.

In [ ]:
def perplexity(probabilities: list[float]) -> float:
    mean_log_probability = sum(log(p) for p in probabilities) / len(probabilities)
    return exp(-mean_log_probability)

print('Model A (effects: 1/2, work: 1/2):', round(perplexity([1 / 2, 1 / 2]), 2))
print('Model B (effects: 1/2, work: 1/8):', round(perplexity([1 / 2, 1 / 8]), 2))

### 3.1 Drei Modelle auf denselben 100 Validierungsreviews

Die Schleife sammelt für jedes tatsächlich gelesene Ziel seine geglättete Wahrscheinlichkeit. Satzenden zählen mit. Die Rechnung mit Logarithmen oben verhindert, dass ein langes Produkt kleiner Zahlen im Rechner auf null fällt. Eine kleinere Perplexität bedeutet, dass das Modell den beobachteten Ziel-Tokens im geometrischen Mittel mehr Wahrscheinlichkeit gibt.

In [ ]:
for n in (1, 2, 3):
    probabilities = []
    for sentence in validation:
        for context, word in word_events(sentence, n):
            counts = models[n].get(context, {})
            p = add_one_probability(counts, word, len(vocabulary))
            probabilities.append(p)
    print(f'{n}-gram: {len(probabilities)} targets, perplexity {perplexity(probabilities):.2f}')

**Deine Auswertung:** Welches Modell erwartet die Wörter dieser 100 Reviews am besten? Warum kann das Trigramm-Modell trotzdem eine überzeugende Fortsetzung für `the special` erzeugen? Beziehe die Zahl der Beobachtungen pro Kontext und die Glättung in deine Antwort ein. [IMDb-Kursdaten](data/imdb/README.md) · [Jurafsky & Martin, *N-gram Language Models*](https://web.stanford.edu/~jurafsky/slp3/3.pdf)